# 03_satellite.ipynb — NASA HLS S30 NDVI Extraction + CDL Corn Masking

**Run on SageMaker. Commit output CSV to repo. Do not re-run locally.**

---

## What This Notebook Does

For each state × year × forecast date:
1. Fetches the USDA Cropland Data Layer (CDL) for that state/year via CropScape API
2. Builds a boolean corn mask (CDL pixel value == 1)
3. Fetches NASA HLS S30 tiles (NIR + Red bands) within ±7 day window
4. Computes NDVI only over corn-masked pixels
5. Averages to a single scalar per state/year/date

Output: `data/raw/ndvi_by_state_date.csv`
Columns: `state`, `year`, `ndvi_aug1`, `ndvi_sep1`, `ndvi_oct1`, `ndvi_final`

---

## States & Bounding Boxes

| State     | West    | South | East    | North |
|-----------|---------|-------|---------|-------|
| Iowa      | -96.64  | 40.37 | -90.14  | 43.50 |
| Colorado  | -109.06 | 36.99 | -102.04 | 41.00 |
| Wisconsin | -92.89  | 42.49 | -86.25  | 47.08 |
| Missouri  | -95.77  | 35.99 | -89.10  | 40.61 |
| Nebraska  | -104.05 | 39.99 | -95.31  | 43.00 |

## Forecast Windows

| Label | Center Date | Window |
|-------|-------------|--------|
| aug1  | August 1    | ±7 days |
| sep1  | September 1 | ±7 days |
| oct1  | October 1   | ±7 days |
| final | November 1  | ±7 days |

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['earthaccess', 'rioxarray', 'pystac-client', 'requests']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies OK')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import requests
import io
import earthaccess
import rioxarray as rxr
from pathlib import Path
from datetime import date, timedelta
from pystac_client import Client

# Earthdata auth — IAM role provides .netrc on SageMaker
try:
    auth = earthaccess.login(strategy='netrc')
    print('Earthdata auth: OK (netrc)')
except Exception:
    auth = earthaccess.login(strategy='interactive')
    print('Earthdata auth: OK (interactive)')

fs = earthaccess.get_fsspec_https_session()

OUT_DIR = Path('../data/raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
STATE_BBOX = {
    'Iowa':      [-96.64, 40.37, -90.14, 43.50],
    'Colorado':  [-109.06, 36.99, -102.04, 41.00],
    'Wisconsin': [-92.89, 42.49, -86.25, 47.08],
    'Missouri':  [-95.77, 35.99, -89.10, 40.61],
    'Nebraska':  [-104.05, 39.99, -95.31, 43.00],
}

FORECAST_DATES = {
    'aug1':  (8, 1),
    'sep1':  (9, 1),
    'oct1':  (10, 1),
    'final': (11, 1),
}
WINDOW_DAYS   = 7
MAX_TILES     = 6
CLOUD_THRESH  = 20   # percent
YEARS         = range(2015, 2025)  # HLS coverage; MODIS handles 2005-2014

# CDL: corn pixel value = 1; clamp to CDL availability
CORN_VALUE    = 1
CDL_YEAR_MIN  = 2008
CDL_YEAR_MAX  = 2024

# HLS STAC
STAC_URL      = 'https://cmr.earthdata.nasa.gov/stac/LPCLOUD'
COLLECTION    = 'HLSS30.v2.0'
NIR_BAND      = 'B8A'
RED_BAND      = 'B04'
FMASK_BAND    = 'Fmask'

catalog = Client.open(STAC_URL, headers={})
print('Config loaded')

In [ ]:
# ── CDL Corn Mask via CropScape API ───────────────────────────────────────────
#
# CropScape is USDA NASS's own REST API. We give it a bounding box + year and
# it returns a GeoTIFF already clipped to that extent, server-side. No new auth,
# no Planetary Computer account, no manual reprojection of a 2GB national file.
#
# CDL pixel value 1 == corn. We return a boolean DataArray: True = corn pixel.

CROPSCAPE_URL = (
    'https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile'
)

# Cache masks so we fetch once per (state, year), not once per (state, year, forecast_date)
_mask_cache = {}

def get_corn_mask(state_name, bbox, year):
    """
    Fetches CDL for the given bbox/year from CropScape and returns a boolean
    DataArray where True == corn pixel. Returns None on fetch failure.
    Results are cached in _mask_cache to avoid redundant API calls.
    """
    cache_key = (state_name, year)
    if cache_key in _mask_cache:
        return _mask_cache[cache_key]

    # CDL only available 2008-2024; clamp accordingly
    cdl_year = min(max(year, CDL_YEAR_MIN), CDL_YEAR_MAX)

    west, south, east, north = bbox
    params = {
        'year':   cdl_year,
        'bbox':   f'{west},{south},{east},{north}',
        'format': 'GTiff',
    }

    try:
        r = requests.get(CROPSCAPE_URL, params=params, timeout=90)
        r.raise_for_status()
        cdl = rxr.open_rasterio(
            io.BytesIO(r.content), masked=True
        ).squeeze()
        mask = (cdl == CORN_VALUE)

        # Safety check — ensure we actually got corn pixels
        n_corn = int(mask.sum().values)
        if n_corn == 0:
            print(f'  WARNING: 0 corn pixels returned for {state_name} CDL {cdl_year}. '
                  f'Mask will be skipped for this state/year.')
            _mask_cache[cache_key] = None
            return None

        print(f'  CDL {cdl_year} loaded for {state_name}: {n_corn:,} corn pixels')
        _mask_cache[cache_key] = mask
        return mask

    except Exception as e:
        print(f'  WARNING: CDL fetch failed for {state_name} {cdl_year}: {e}. '
              f'Proceeding without mask.')
        _mask_cache[cache_key] = None
        return None

print('CDL helpers defined')

In [ ]:
# ── HLS Tile Search + NDVI Computation ───────────────────────────────────────

def search_hls_tiles(bbox, month, day, year, window=WINDOW_DAYS):
    """Search HLS STAC for tiles within ±window days of the target date."""
    target = date(year, month, day)
    start  = (target - timedelta(days=window)).isoformat()
    end    = (target + timedelta(days=window)).isoformat()
    results = catalog.search(
        collections=[COLLECTION],
        bbox=bbox,
        datetime=f'{start}/{end}',
        max_items=MAX_TILES,
    )
    return list(results.items())


def compute_ndvi_for_tile(item, corn_mask=None):
    """
    Computes mean corn-masked NDVI for one HLS tile.
    Falls back to full-tile NDVI if corn_mask is None or produces no overlap.
    Returns np.nan on read/computation error.
    """
    try:
        # Load NIR and Red bands
        with fs.open(item.assets[NIR_BAND].href) as f:
            nir = rxr.open_rasterio(f, masked=True).squeeze().astype(float)
        with fs.open(item.assets[RED_BAND].href) as f:
            red = rxr.open_rasterio(f, masked=True).squeeze().astype(float)

        # Compute NDVI
        ndvi = (nir - red) / (nir + red)
        ndvi = ndvi.where((ndvi >= -1) & (ndvi <= 1))

        # Apply corn mask if available
        if corn_mask is not None:
            try:
                # reproject_match warps CDL mask to match this tile's CRS/grid
                mask_reproj = corn_mask.rio.reproject_match(ndvi)
                n_corn = int(mask_reproj.sum().values)
                if n_corn > 0:
                    ndvi = ndvi.where(mask_reproj)
                else:
                    # Tile doesn't overlap corn pixels — skip mask, use full tile
                    print(f'    No corn overlap for tile {item.id} — using full tile NDVI')
            except Exception as e:
                print(f'    Mask reproject failed for tile {item.id}: {e} — using full tile NDVI')

        val = float(ndvi.mean().values)
        return val if not np.isnan(val) else np.nan

    except Exception as e:
        print(f'    Tile read error ({item.id}): {e}')
        return np.nan


print('HLS helpers defined')

In [ ]:
# ── Validation: Smoke-test Iowa 2022 Aug1 before full run ────────────────────
#
# Run this cell first. Confirm corn pixel count is nonzero and NDVI is in
# [0.4, 0.9] range (healthy corn in August). If either check fails, debug
# before committing to the full multi-hour run.

print('=== SMOKE TEST: Iowa 2022 Aug1 ===')

test_state = 'Iowa'
test_bbox  = STATE_BBOX[test_state]
test_year  = 2022
test_month, test_day = FORECAST_DATES['aug1']

# CDL mask
test_mask = get_corn_mask(test_state, test_bbox, test_year)

# HLS tiles
test_tiles = search_hls_tiles(test_bbox, test_month, test_day, test_year)
print(f'HLS tiles found: {len(test_tiles)}')

if test_tiles:
    test_ndvi = compute_ndvi_for_tile(test_tiles[0], corn_mask=test_mask)
    print(f'Corn-masked NDVI (first tile): {test_ndvi:.4f}')
    assert 0.3 < test_ndvi < 1.0, (
        f'NDVI {test_ndvi:.4f} out of expected range for Iowa corn in August. '
        f'Check mask or band selection.'
    )
    print('Smoke test PASSED — proceeding to full run.')
else:
    print('WARNING: No HLS tiles found for smoke test. Check STAC query and date window.')

In [ ]:
# ── Main Extraction Loop ──────────────────────────────────────────────────────
#
# For each state × year × forecast date:
#   1. Fetch corn mask (cached per state/year — fetched once, reused 4x)
#   2. Search HLS tiles in date window
#   3. Compute corn-masked NDVI for each tile
#   4. Median across tiles (robust to outlier tiles)
#   5. Record result

records = []

for state, bbox in STATE_BBOX.items():
    print(f'\n{"="*60}')
    print(f'STATE: {state}')
    print(f'{"="*60}')

    for year in YEARS:
        print(f'\n  Year: {year}')

        # Fetch corn mask once per (state, year) — reused across all 4 forecast dates
        corn_mask = get_corn_mask(state, bbox, year)

        for label, (month, day) in FORECAST_DATES.items():
            tiles = search_hls_tiles(bbox, month, day, year)

            if not tiles:
                print(f'    {label}: no tiles found')
                records.append({
                    'state': state.upper(), 'year': year,
                    'forecast_date': label, 'ndvi_mean': np.nan,
                    'n_tiles': 0, 'corn_masked': corn_mask is not None
                })
                continue

            ndvi_vals = [
                compute_ndvi_for_tile(tile, corn_mask=corn_mask)
                for tile in tiles
            ]
            ndvi_vals = [v for v in ndvi_vals if not np.isnan(v)]

            ndvi = float(np.median(ndvi_vals)) if ndvi_vals else np.nan
            masked_str = 'corn-masked' if corn_mask is not None else 'unmasked'
            print(f'    {label}: {len(ndvi_vals)}/{len(tiles)} tiles OK | '
                  f'NDVI={ndvi:.4f} ({masked_str})')

            records.append({
                'state': state.upper(), 'year': year,
                'forecast_date': label, 'ndvi_mean': ndvi,
                'n_tiles': len(ndvi_vals),
                'corn_masked': corn_mask is not None
            })

print(f'\nDone. {len(records)} records collected.')

In [ ]:
# ── Reshape to Wide Format + Forward-fill Missing Windows ─────────────────────

long_df = pd.DataFrame(records)

# Diagnostic: how many windows had no tiles?
missing = long_df[long_df['ndvi_mean'].isna()]
print(f'Missing windows (no tiles or all-NaN): {len(missing)}')
if len(missing):
    print(missing[['state','year','forecast_date']].to_string(index=False))

# Diagnostic: how many windows used the corn mask?
masked_pct = long_df['corn_masked'].mean() * 100
print(f'\nWindows with CDL corn mask applied: {masked_pct:.1f}%')

wide_df = (
    long_df
    .pivot_table(index=['state','year'], columns='forecast_date', values='ndvi_mean')
    .reset_index()
)
wide_df.columns.name = None
wide_df = wide_df.rename(columns={
    'aug1':  'ndvi_aug1',
    'sep1':  'ndvi_sep1',
    'oct1':  'ndvi_oct1',
    'final': 'ndvi_final',
})

# Forward-fill cloudy/missing windows within each state/year row
ndvi_cols = ['ndvi_aug1', 'ndvi_sep1', 'ndvi_oct1', 'ndvi_final']
wide_df[ndvi_cols] = wide_df[ndvi_cols].interpolate(axis=1, limit_direction='both')

print(f'\nFinal shape: {wide_df.shape}')
print(wide_df.head(10))

In [ ]:
# ── Save Output ───────────────────────────────────────────────────────────────

out_path = OUT_DIR / 'ndvi_by_state_date.csv'
wide_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Rows: {len(wide_df)} | Years: {wide_df.year.min()}–{wide_df.year.max()}')
print(f'\nCoverage by state:')
print(wide_df.groupby('state')[['year']].agg(['min','max','count']))
print(f'\nNDVI summary stats:')
print(wide_df[ndvi_cols].describe().round(4))
print('\nCommit ndvi_by_state_date.csv to repo. Do not re-run this notebook locally.')